In [1]:
!pip install streamlit python-dotenv openai pydantic fastmcp pydantic-ai arxiv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 12.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-instrumentation to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opentelemetry-instrumentation-httpx to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of opentelemetry-instrumentation-httpx to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver

In [2]:
import streamlit as st
import dotenv
import urllib, urllib.request
import os
from openai import OpenAI
from pydantic import BaseModel, Field
import json
from fastmcp import FastMCP
from pydantic_ai.toolsets.fastmcp import FastMCPToolset
import arxiv

st.set_page_config(layout="wide")

mcp = FastMCP("arXiv-Searcher")

def search_arxiv(query: str, max_results: int) -> str:
    """
    Search the arXiv preprint repository for academic papers and download their PDFs.
    Returns a JSON string containing the matching papers.
    """
    if not query or not str(query).strip():
        return json.dumps({"error": "The search query cannot be empty. Please provide a valid query string."})

    client = arxiv.Client()

    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance
    )

    results = []
    os.makedirs("downloads", exist_ok=True)

    for paper in client.results(search):
        safe_name = "".join([c for c in paper.title if c.isalpha() or c.isdigit() or c==' ']).rstrip()
        filename = f"{safe_name.replace(' ', '_')}.pdf"
        filepath = os.path.join("downloads", filename)

        try:
            paper.download_pdf(dirpath="downloads", filename=filename)
            download_status = "Success"
        except Exception as e:
            print(f"Failed to download {filename}: {e}")
            download_status = f"Failed to download locally: {str(e)}"

        paper_data = {
            "id": paper.get_short_id(),
            "title": paper.title,
            "authors": [author.name for author in paper.authors],
            "summary": paper.summary.replace('\n', ' '),
            "pdf_url": paper.pdf_url,
            "local_pdf_path": filepath,
            "download_status": download_status
        }
        results.append(paper_data)

    if not results:
        return json.dumps({"error": f"No papers found on arXiv for the query: '{query}'"})

    return json.dumps(results, indent=2)


st.markdown(
    """
    <style>
    .stApp {
        background-color: #808088   ;
    }
    </style>
    """,
    unsafe_allow_html=True
)

def chat_page():
    st.title("Research System")

    dotenv.load_dotenv()

    model_name = 'llama3.2:1b'

    if "messages" not in st.session_state:
        st.session_state.messages = [
            {"role": "system", "content":
             "You are a helpful research assistant."
             "You MUST use the tools provided if the user asks to use them"
            },
        ]

    with st.spinner("Thinking..."):
        client = OpenAI(
            base_url='http://localhost:11434/v1/',
            api_key='ollama',
        )
        for message in st.session_state.messages:
            if message["role"] not in ["system", "tool"] and message.get("content"):
                with st.chat_message(message["role"]):
                    st.markdown(message["content"])

    tools =  [{
        "type": "function",
        "function": {
            "name": "search_arxiv",
            "description": "searches in arxiv for papers and downloads their PDFs to local disk",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "the query to search for on arxiv",
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "the maximum number of results to return",
                    }
                },
                "required": ["query", "max_results"],
            },
        },
    }]
    pormpt = st.chat_input("Ask about something", key="chat_input")
    if pormpt:
        st.session_state.messages.append({"role": "user", "content": pormpt})
        with st.chat_message("user"):
            st.markdown(pormpt)

        response = client.chat.completions.create(
            model=model_name,
            messages=st.session_state.messages,
            tools=tools,
            tool_choice="auto",
        )

        response_message = response.choices[0].message

        assistant_dict = {"role": "assistant", "content": response_message.content}
        if response_message.tool_calls:
            assistant_dict["tool_calls"] = [
                {
                    "id": tc.id,
                    "type": tc.type,
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments
                    }
                } for tc in response_message.tool_calls
            ]
        st.session_state.messages.append(assistant_dict)

        if response_message.content:
            with st.chat_message("assistant"):
                st.markdown(response_message.content)

        if response_message.tool_calls:
            for tool_call in response_message.tool_calls:
                if tool_call.function.name == "search_arxiv":
                    try:
                        args = json.loads(tool_call.function.arguments)
                    except json.JSONDecodeError:
                        args = {}
                        st.error("The AI attempted to search but provided malformed arguments. Try asking again.")

                    query = args.get("query", "")
                    try:
                        max_results = int(args.get("max_results", 1))
                    except (ValueError, TypeError):
                        max_results = 1

                    with st.chat_message("assistant"):
                        st.markdown(f"*(Searching arXiv for up to {max_results} paper(s) for query: {query}...)*")

                    tool_result = search_arxiv(query=query, max_results=max_results)

                    st.session_state.messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": tool_call.function.name,
                        "content": tool_result
                    })
                else:
                    st.session_state.messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": tool_call.function.name,
                        "content": json.dumps({"error": f"Unknown tool: {tool_call.function.name}"})
                    })

            final_response = client.chat.completions.create(
                model=model_name,
                messages=st.session_state.messages,
            )
            final_msg = final_response.choices[0].message.content
            st.session_state.messages.append({"role": "assistant", "content": final_msg})
            with st.chat_message("assistant"):
                st.markdown(final_msg)

pages = [
    st.Page(chat_page, title="Chat", default=True),
    st.Page("Files.py", title="Files")
]

page = st.navigation(pages)
page.run()

2026-05-19 20:52:16.367 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-19 20:52:16.373 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-19 20:52:16.551 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-05-19 20:52:16.552 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-19 20:52:16.553 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-19 20:52:16.555 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-19 20:52:16.557 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

In [3]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 30.8 MB/s eta 0:00:00


In [4]:
import streamlit as st
import os
import pypdf
import re
from openai import OpenAI
import concurrent.futures
from pydantic import BaseModel, Field
import json

class HighlightResult(BaseModel):
    phrases: list[str] = Field(description="A list of exact phrases or sentences to highlight from the text.")

DOWNLOADS_DIR = "downloads"

@st.cache_data(show_spinner="Generating summary...", persist="disk")
def get_summary(text):
    client = OpenAI(base_url='http://localhost:11434/v1/', api_key='ollama')
    response = client.chat.completions.create(
        model="llama3.2:1b",
        messages=[
            {"role": "system", "content": "You are an expert academic research assistant. Your task is to provide a concise, high-level summary of the main themes, objectives, and findings of the provided text. Keep the summary clear, brief, and informative. Do NOT include any conversational filler or introductions."},
            {"role": "user", "content": text[:5000]}
        ],
        temperature=0.3
    )
    return response.choices[0].message.content

@st.cache_data(show_spinner=False, persist="disk")
def get_highlighted_chunk(chunk, summary_text):
    client = OpenAI(base_url='http://localhost:11434/v1/', api_key='ollama')
    response = client.chat.completions.create(
        model="llama3.2:1b",
        messages=[
            {"role": "system", "content": f"You are an expert reading assistant. Your task is to extract the most important key phrases or crucial sentences from the provided text. "
                                f"You MUST output your response in JSON format. The JSON must contain a single key 'phrases' containing a list of strings. "
                                f"The phrases MUST be exact substrings from the provided text. "
                                f"Use this summary of the document's theme to guide your highlights: {summary_text}"},
            {"role": "user", "content": chunk}
        ],
        temperature=0.3,
        response_format={"type": "json_object"}
    )

    highlighted_chunk = chunk
    try:
        result = HighlightResult.model_validate_json(response.choices[0].message.content)
        phrases_to_highlight = result.phrases

        for phrase in phrases_to_highlight:
            phrase = phrase.strip()
            if phrase and len(phrase) > 4 and phrase in highlighted_chunk:
                highlighted_chunk = highlighted_chunk.replace(phrase, f"<mark>{phrase}</mark>")
    except Exception as e:
        # Fallback in case the local LLM fails to return valid JSON
        print(f"Failed to parse JSON for chunk: {e}")

    return highlighted_chunk

@st.cache_data(show_spinner="Highlighting page...", persist="disk")
def get_highlighted_page(cleaned_text, summary_text):
    sentences = re.split(r'(?<=[.!?])\s+', cleaned_text)
    chunks = []
    current_chunk = ""
    for sentence in sentences:
        if len((current_chunk + sentence).split()) > 500 and current_chunk:
            chunks.append(current_chunk.strip())
            current_chunk = sentence + " "
        else:
            current_chunk += sentence + " "
    if current_chunk:
        chunks.append(current_chunk.strip())

    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = [executor.submit(get_highlighted_chunk, chunk, summary_text) for chunk in chunks]
        highlighted_chunks = [future.result() for future in futures]

    return " ".join(highlighted_chunks)

def Reading_page(pdf_file):
    st.session_state.selected_pdf = pdf_file
    file_path = os.path.join(DOWNLOADS_DIR, st.session_state.selected_pdf)
    summary_text = ""

    reader = pypdf.PdfReader(file_path)
    raw_text = ""
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            raw_text += text + "\n"

    col1, col2, col3 = st.columns([1, 2, 1])
    with col1:
        if st.button("Back to files list"):
            st.session_state.selected_pdf = None
            st.rerun()
        st.markdown("### Summary:")

        summary_text = get_summary(raw_text)
        st.markdown(summary_text)
        st.markdown("---")

    with col2:
        ai_highlight = st.toggle("AI Highlighting")

        if ai_highlight and st.button("Redo Highlighting"):
            get_highlighted_chunk.clear()

        if os.path.exists(file_path):
            st.subheader(f"Reading: {st.session_state.selected_pdf.replace('_', ' ').replace('.pdf', '')}")
            try:
                for i, page in enumerate(reader.pages):
                    raw_text = page.extract_text()
                    if raw_text:
                        cleaned_text = re.sub(r'\s+', ' ', raw_text).strip()

                        if cleaned_text:
                            sentences = re.split(r'(?<=[.!?])\s+', cleaned_text)
                            chunks = []
                            current_chunk = ""
                            for sentence in sentences:
                                if len((current_chunk + sentence).split()) > 500 and current_chunk:
                                    chunks.append(current_chunk.strip())
                                    current_chunk = sentence + " "
                                else:
                                    current_chunk += sentence + " "
                            if current_chunk:
                                chunks.append(current_chunk.strip())

                            if ai_highlight:
                                with st.spinner("Highlighting page..."):
                                    with concurrent.futures.ThreadPoolExecutor() as executor:
                                        futures = [executor.submit(get_highlighted_chunk, chunk, summary_text) for chunk in chunks]
                                        highlighted_chunks = [future.result() for future in futures]
                            else:
                                highlighted_chunks = chunks
                            cleaned_text = " ".join(highlighted_chunks)

                        st.markdown(cleaned_text, unsafe_allow_html=True)
                        if i < len(reader.pages) - 1:
                            st.markdown("---")
            except Exception as e:
                st.error(f"Error extracting text: {e}")
        else:
            st.error("File not found.")

2026-05-19 20:52:23.141 No runtime found, using MemoryCacheStorageManager
2026-05-19 20:52:23.142 No runtime found, using MemoryCacheStorageManager
2026-05-19 20:52:23.144 No runtime found, using MemoryCacheStorageManager


In [ ]:
!streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py




2026-05-19 20:53:05.374 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.126.107.90:8501

